In [ ]:
import os
import mlflow
from mlflow.tracking import MlflowClient
import pandas as pd
from datetime import datetime

# Get MLflow configuration from environment variables
MLFLOW_TRACKING_URI= os.environ.get("MLFLOW_TRACKING_URI", "")
MLFLOW_EXPERIMENT_NAME= os.environ.get("MLFLOW_EXPERIMENT_NAME", "")
MLFLOW_TRACKING_PASSWORD= os.environ.get("MLFLOW_TRACKING_PASSWORD", "")
MLFLOW_TRACKING_USERNAME= os.environ.get("MLFLOW_TRACKING_USERNAME", "")
print(f"MLflow Tracking URI: {MLFLOW_TRACKING_URI}")
print(f"Experiment Name: {MLFLOW_EXPERIMENT_NAME}")
print(f"Password: {MLFLOW_TRACKING_PASSWORD}")

In [ ]:
# Set up MLflow client
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
client = MlflowClient()

# Get experiment by name
try:
    experiment = mlflow.get_experiment_by_name(MLFLOW_EXPERIMENT_NAME)
    print(f"Experiment: {experiment}")
    if experiment is None:
        raise ValueError(f"Experiment '{MLFLOW_EXPERIMENT_NAME}' not found")
    
    experiment_id = experiment.experiment_id
    print(f"Found experiment: {experiment.name} (ID: {experiment_id})")
except Exception as e:
    print(f"Error getting experiment: {e}")
    experiment_id = None

In [ ]:
# Search for traces in the experiment
if experiment_id:
    try:
        traces = client.search_traces(
            experiment_ids=[experiment_id],
            max_results=100,
            order_by=["timestamp_ms DESC"]
        )
        
        print(f"\nFound {len(traces)} traces\n")
        
        # Convert traces to a more readable format
        trace_data = []
        for trace in traces:
            trace_info = {
                "trace_id": trace.info.request_id,
                "timestamp": datetime.fromtimestamp(trace.info.timestamp_ms / 1000),
                "status": trace.info.status,
                "execution_time_ms": trace.info.execution_time_ms,
                "tags": trace.info.tags if hasattr(trace.info, 'tags') else {}
            }
            trace_data.append(trace_info)
        
        # Display as DataFrame
        if trace_data:
            df_traces = pd.DataFrame(trace_data)
            display(df_traces)
        else:
            print("No traces found in this experiment")
            
    except Exception as e:
        print(f"Error searching traces: {e}")
        traces = []
else:
    print("Cannot search traces without valid experiment_id")